In [1]:
import pandas as pd
from credit_risk.data.registry import build_default_registry
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)

In [2]:
registry = build_default_registry()
accepted_dataset = registry.get('lending_club_accepted_raw')

In [3]:
df = pd.read_csv(accepted_dataset.path)


/var/folders/_y/z6qlgwb57vz2l3q1xqw5zb200000gn/T/ipykernel_51944/3650065708.py:1: DtypeWarning: Columns (0: id, 1: desc, 2: next_pymnt_d, 3: verification_status_joint, 4: sec_app_earliest_cr_line, 5: hardship_type, 6: hardship_reason, 7: hardship_status, 8: hardship_start_date, 9: hardship_end_date, 10: payment_plan_start_date, 11: hardship_loan_status, 12: debt_settlement_flag_date, 13: settlement_status, 14: settlement_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(accepted_dataset.path)


## Creating data dictionary with descriptions

In [ ]:
# Build a column-level data dictionary using Lending Club's official definitions.
# Source: https://resources.lendingclub.com/LCDataDictionary.xlsx
import re

DATA_DICTIONARY_URL = "https://resources.lendingclub.com/LCDataDictionary.xlsx"

loanstats_dict = pd.read_excel(DATA_DICTIONARY_URL, sheet_name="LoanStats")
browsenotes_dict = pd.read_excel(DATA_DICTIONARY_URL, sheet_name="browseNotes")


def normalize_name(name: str) -> str:
    """Normalize dictionary and dataset field names to a common format for robust matching."""
    normalized = str(name).replace("\xa0", " ").strip().lower()
    normalized = re.sub(r"(?<!^)(?=[A-Z])", "_", normalized)
    normalized = normalized.replace(" ", "_").replace("-", "_").replace("/", "_")
    normalized = re.sub(r"[^a-z0-9_]+", "", normalized)
    normalized = re.sub(r"_+", "_", normalized).strip("_")
    return normalized


dictionary_rows = pd.concat(
    [
        loanstats_dict.rename(columns={"LoanStatNew": "dictionary_field", "Description": "description"})[
            ["dictionary_field", "description"]
        ],
        browsenotes_dict.rename(columns={"BrowseNotesFile": "dictionary_field", "Description": "description"})[
            ["dictionary_field", "description"]
        ],
    ],
    ignore_index=True,
)

normalized_description_map = {}
for _, row in dictionary_rows.dropna(subset=["dictionary_field", "description"]).iterrows():
    key = normalize_name(row["dictionary_field"])
    if key not in normalized_description_map:
        normalized_description_map[key] = str(row["description"]).strip()

# Official file uses `verified_status_joint`, while dataset column is `verification_status_joint`.
normalized_description_map["verification_status_joint"] = normalized_description_map.get(
    "verified_status_joint"
)


column_info = pd.DataFrame({"column": df.columns})
column_info["normalized_column"] = column_info["column"].map(normalize_name)
column_info["description"] = column_info["normalized_column"].map(normalized_description_map)
column_info["dtype"] = column_info["column"].map(df.dtypes.astype(str).to_dict())
column_info["missing_percentage"] = (
    column_info["column"].map(df.isna().mean().mul(100).round(2).to_dict())
)
column_info["unique_count"] = column_info["column"].map(df.nunique(dropna=True).to_dict())
column_info["example_values"] = column_info["column"].map(
    lambda col: df[col].dropna().drop_duplicates().head(5).tolist()
)

missing_description_columns = column_info.loc[column_info["description"].isna(), "column"].tolist()
if missing_description_columns:
    raise ValueError(
        "Missing official descriptions for columns: " + ", ".join(missing_description_columns)
    )

data_dictionary_df = column_info[
    [
        "column",
        "description",
        "dtype",
        "missing_percentage",
        "unique_count",
        "example_values",
    ]
].copy()

print(f"Data dictionary ready: {len(data_dictionary_df)} columns documented.")

data_dictionary_df.head(20)

In [21]:
data_dictionary_df.to_csv('../artifacts/reports/lending_club_accepted_dictionary.csv')

## Initial data cleaning

In [10]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(1), str(37)
memory usage: 5.8 GB


In [ ]:
cols_to_discard = [
    "id",
    "member_id",
    "",
]


In [5]:
df['home_ownership'].value_counts()

home_ownership
MORTGAGE    1111450
RENT         894929
OWN          253057
ANY             996
OTHER           182
NONE             54
Name: count, dtype: int64